# 01: Data lake inventory

**Phase 1: what have I actually got?**

A tally of the lake — databases, tables, row counts, partition ranges, date
coverage, site and circuit counts — and the **real** schema of the candidate
tables, read from the catalogue rather than inferred from what the pipeline code
implies they contain.

---

### Expected Athena scan

**Effectively zero bytes of DATA.** Everything below reads metadata:

| Source | What it gives | Data scanned |
|---|---|---|
| Glue API | every table, its format, location, columns, partition keys | none — not an Athena query |
| `information_schema.columns` | the schema Athena actually resolves | none |
| `"table$partitions"` (Iceberg) | **exact per-partition row counts and byte sizes** | manifests only, ~0 |
| dimension tables | site and circuit counts | a few MB |
| partition-filtered `LIMIT 5` | what a row looks like | one partition, a handful of columns |

The one that matters is `$partitions`. On an Iceberg table it returns exact row
counts and file sizes straight from the manifests, so the entire "how big is `ts`"
question is answered without reading a single data file. If `ts` turns out to be
Hive rather than Iceberg, `$partitions` gives partition *values* only and the row
counts have to be bought — the notebook says so where that happens and does not
buy them silently.

Athena's 10 MB per-query minimum dominates: roughly 25 queries × 10 MB ≈ **250 MB
billable, about AUD 0.002.** Section 9 prints the measured figure.

**Nothing in this notebook scans more than a few GB. If any query does, the guard
in `ami_athena` has been bypassed and that is a bug.**


## Setup


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from bms_sa_review.ami_data_analysis.config import ami_config as Config
from bms_sa_review.ami_data_analysis.lib import ami_athena as Athena
from bms_sa_review.ami_data_analysis.lib import ami_inventory as Inventory

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

Athena.reset_scan_log()
Athena.require_credentials()
print("Credentials OK. Starting inventory.")


Credentials OK. Starting inventory.


## 1. The catalogue

- `is_iceberg` is the column to read first. 
- An Iceberg table exposes `$partitions`,`$files`, `$snapshots` and `$history`, which is how the next section gets row counts for nothing. A Hive table does not.


In [2]:
catalog = Inventory.glue_inventory()
print(f"{len(catalog)} tables across {catalog.database.nunique()} databases")
display(Inventory.database_summary(catalog))


55 tables across 6 databases


,database,n_tables,n_iceberg,n_partitioned
0,solar_analytics_iceberg,35,35,0
1,solar_analytics,13,0,4
2,test_db,3,3,0
3,sapn2022,2,2,0
4,elb_logdb,1,0,1
5,bom_nci,1,1,0


In [3]:
display(catalog[["database", "table", "table_type", "format", "is_iceberg",
                 "n_columns", "partition_keys", "updated"]])


,database,table,table_type,format,is_iceberg,n_columns,partition_keys,updated
0,bom_nci,solar,EXTERNAL_TABLE,ICEBERG,True,14,,2026-06-20 18:18:52+10:00
1,elb_logdb,elb_logs_tbl,EXTERNAL_TABLE,EXTERNAL_TABLE,False,32,"year, month, day",2023-03-23 18:43:31+11:00
2,sapn2022,circuit_measurements,EXTERNAL_TABLE,ICEBERG,True,10,,2026-05-21 17:27:25+10:00
3,sapn2022,circuit_measurements_curtailment_train,EXTERNAL_TABLE,ICEBERG,True,10,,2026-05-22 07:39:23+10:00
4,solar_analytics,circuits,EXTERNAL_TABLE,EXTERNAL_TABLE,False,7,,2025-10-01 15:28:56+10:00
5,solar_analytics,compliance_voltvar,EXTERNAL_TABLE,EXTERNAL_TABLE,False,19,,2025-10-01 15:28:55+10:00
6,solar_analytics,compliance_voltwatt,EXTERNAL_TABLE,EXTERNAL_TABLE,False,8,,2025-10-01 15:28:54+10:00
7,solar_analytics,meta_single_inverters,EXTERNAL_TABLE,EXTERNAL_TABLE,False,22,,2025-10-01 15:28:53+10:00
8,solar_analytics,meta_single_inverters_wrong_capacity,EXTERNAL_TABLE,EXTERNAL_TABLE,False,37,,2025-10-01 15:28:53+10:00
9,solar_analytics,meta_single_inverters_wrong_capacity_up2_3c,EXTERNAL_TABLE,EXTERNAL_TABLE,False,41,,2025-10-01 15:28:56+10:00


### Storage locations

Where the files physically are. Two tables pointing at the same prefix, or a table
whose location is somewhere unexpected, is worth checking


In [4]:
display(catalog[["database", "table", "location"]].sort_values(["location"])
        .reset_index(drop=True))


,database,table,location
0,elb_logdb,elb_logs_tbl,s3://elb-accesslogs-130340360668-ap-southeast-...
1,solar_analytics,raw_bom_2024_6,s3://project-ciccada/BOM_NCI/2024/6/
2,bom_nci,solar,s3://project-ciccada/Trino-Warehouse/BOM_NCI/s...
3,sapn2022,circuit_measurements,s3://project-ciccada/Trino-Warehouse/SAPN2022/...
4,sapn2022,circuit_measurements_curtailment_train,s3://project-ciccada/Trino-Warehouse/SAPN2022/...
5,test_db,evm_batch_1ee9f79bdf91,s3://project-ciccada/Trino-Warehouse/Test_DB/e...
6,test_db,evm_batch_a4f103335e90,s3://project-ciccada/Trino-Warehouse/Test_DB/e...
7,test_db,evm_batch_d334662bbf60,s3://project-ciccada/Trino-Warehouse/Test_DB/e...
8,solar_analytics_iceberg,all_uncurtailedpv,s3://project-ciccada/Trino-Warehouse/solar_ana...
9,solar_analytics_iceberg,all_uncurtailedpv_lso,s3://project-ciccada/Trino-Warehouse/solar_ana...


## 2. Real schemas

- `information_schema.columns` is what **Athena** will let you select. 
- That is not always what Glue declares
- `aws_config.describe()` notes that the Glue column list can come back empty for Iceberg tables, which is why `n_columns` above may disagree with what you see here.

One query per database rather than one `DESCRIBE` per table.

In [5]:
columns = pd.concat(
    [Inventory.column_inventory(db).assign(database=db) for db in (Config.SA, Config.SAI)],
    ignore_index=True,
)
print(f"{len(columns):,} column definitions across {columns.table_name.nunique()} tables")

width = (columns.groupby(["database", "table_name"], as_index=False)
                .agg(n_columns=("column_name", "count")))
display(width.sort_values("n_columns", ascending=False).reset_index(drop=True))


672 column definitions across 46 tables


,database,table_name,n_columns
0,solar_analytics,meta_single_inverters_wrong_capacity_up2_3c,41
1,solar_analytics_iceberg,meta_up23c,38
2,solar_analytics,meta_single_inverters_wrong_capacity,37
3,solar_analytics_iceberg,conformance_voltvar_v2,34
4,solar_analytics_iceberg,conformance_voltvar_v2_flex_included,34
5,solar_analytics_iceberg,conformance_voltvar,23
6,solar_analytics_iceberg,structured_data_v2,23
7,solar_analytics_iceberg,structured_data_v2_flex_included,23
8,solar_analytics,meta_single_inverters,22
9,solar_analytics,compliance_voltvar,19


## 3. Partition metadata

- For every table that declares partition keys, read its `$partitions` metadata table. 
- On Iceberg this returns exact record counts, file counts and total bytes per partition, from the manifests, without touching data.
- Tables that will not serve `$partitions` are logged and skipped


In [6]:
totals, raw_partitions, partition_log = Inventory.probe_partitions(catalog)
print()
display(partition_log)


  bom_nci.solar                                        384 partitions,   8,882,155,830 rows  [partitioning hidden from Glue]
  elb_logdb.elb_logs_tbl                             29760 partitions
  sapn2022.circuit_measurements                          1 partitions,     364,427,972 rows
  sapn2022.circuit_measurements_curtailment_train        1 partitions,     436,881,072 rows
  solar_analytics.test_sola_2025_12                    184 partitions
  solar_analytics.test_sola_2025_7                     184 partitions
  solar_analytics.test_sola_2025_8                     184 partitions
  solar_analytics.test_sola_2025_9                     184 partitions
  solar_analytics_iceberg.all_uncurtailedpv             24 partitions,     590,219,662 rows  [partitioning hidden from Glue]
  solar_analytics_iceberg.all_uncurtailedpv_lso         24 partitions,      37,301,774 rows  [partitioning hidden from Glue]
  solar_analytics_iceberg.all_uncurtailedpv_v2          24 partitions,     487,222,740 rows

,table,ok,n_partitions,declared_partition_keys,actual_partition_columns,glue_hides_partitioning,error
0,bom_nci.solar,True,384,(none declared in Glue),"year, month",True,
1,elb_logdb.elb_logs_tbl,True,29760,"year, month, day","year, month, day",False,
2,sapn2022.circuit_measurements,True,1,(none declared in Glue),,False,
3,sapn2022.circuit_measurements_curtailment_train,True,1,(none declared in Glue),,False,
4,solar_analytics.test_sola_2025_12,True,184,"year, month, day","year, month, day",False,
5,solar_analytics.test_sola_2025_7,True,184,"year, month, day","year, month, day",False,
6,solar_analytics.test_sola_2025_8,True,184,"year, month, day","year, month, day",False,
7,solar_analytics.test_sola_2025_9,True,184,"year, month, day","year, month, day",False,
8,solar_analytics_iceberg.all_uncurtailedpv,True,24,(none declared in Glue),"year, month",True,
9,solar_analytics_iceberg.all_uncurtailedpv_lso,True,24,(none declared in Glue),"year, month",True,


### What `$partitions` actually returned

Shown raw, before anything interprets it. The column set differs between Athena
engine versions and between Iceberg and Hive, and `normalise_partitions()` is
written to be tolerant of that — but it is worth seeing the real shape once,
because Phase 4's size estimate is built on these columns.


In [7]:
ts_key = next((k for k in raw_partitions if k.endswith(".ts")), None)
if ts_key is None:
    print("No `ts` partition metadata was returned. Available keys:")
    for key in sorted(raw_partitions):
        print("  -", key)
else:
    print(f"raw {ts_key}$partitions -- {len(raw_partitions[ts_key])} rows, "
          f"columns: {list(raw_partitions[ts_key].columns)}")
    display(raw_partitions[ts_key].head(10))


raw solar_analytics_iceberg.ts$partitions -- 816 rows, columns: ['partition', 'record_count', 'file_count', 'total_size', 'data']


,partition,record_count,file_count,total_size,data
0,"{year=2024, month=12, is_pv=false, postcode_bu...",12931038,9,366565261,"{circuit_id={min=222752, max=674304, null_coun..."
1,"{year=2024, month=12, is_pv=false, postcode_bu...",20218675,9,669146890,"{circuit_id={min=6275, max=657082, null_count=..."
2,"{year=2025, month=10, is_pv=false, postcode_bu...",24647323,2,739665395,"{circuit_id={min=208830, max=705014, null_coun..."
3,"{year=2025, month=10, is_pv=false, postcode_bu...",14761188,1,416179096,"{circuit_id={min=6367, max=708649, null_count=..."
4,"{year=2025, month=10, is_pv=false, postcode_bu...",23896296,2,697318288,"{circuit_id={min=10296, max=699807, null_count..."
5,"{year=2025, month=10, is_pv=false, postcode_bu...",18689636,1,529874432,"{circuit_id={min=6067, max=699958, null_count=..."
6,"{year=2025, month=10, is_pv=false, postcode_bu...",18490723,1,525921571,"{circuit_id={min=204633, max=708505, null_coun..."
7,"{year=2025, month=10, is_pv=false, postcode_bu...",20974470,1,573733524,"{circuit_id={min=24822, max=704621, null_count..."
8,"{year=2025, month=10, is_pv=false, postcode_bu...",30866163,2,942238384,"{circuit_id={min=15251, max=702551, null_count..."
9,"{year=2025, month=10, is_pv=false, postcode_bu...",21313649,1,610059888,"{circuit_id={min=66593, max=704824, null_count..."


## 4. The one-screen summary

Everything above, joined. Sorted by row count, so the fact tables sort to the top
and the dimension tables fall to the bottom.

Blank `n_rows` means the table was not probed (unpartitioned) or `$partitions`
carried no counts (Hive) — **not** that the table is empty.


In [8]:
summary = Inventory.summary_table(catalog, totals)
display(summary)


,database,table,fmt,cols,partitioned_by,n_partitions,n_rows,size,coverage,is_pv
0,solar_analytics_iceberg,ts,iceberg,16,"year, month, is_pv [not declared in Glue]",816.0,1.634525e+10,456.93 GB,2024-01 .. 2025-12,"false, true"
1,bom_nci,solar,iceberg,14,"year, month [not declared in Glue]",384.0,8.882156e+09,73.38 GB,2024-01 .. 2025-12,
2,solar_analytics_iceberg,structured_data,iceberg,18,"year, month [not declared in Glue]",24.0,1.022901e+09,27.83 GB,2024-01 .. 2025-12,
3,solar_analytics_iceberg,structured_data_v2_flex_included,iceberg,23,"year, month [not declared in Glue]",24.0,8.716554e+08,36.62 GB,2024-01 .. 2025-12,
4,solar_analytics_iceberg,structured_data_v2,iceberg,23,"year, month [not declared in Glue]",24.0,8.414910e+08,35.38 GB,2024-01 .. 2025-12,
5,solar_analytics_iceberg,all_uncurtailedpv,iceberg,8,"year, month [not declared in Glue]",24.0,5.902197e+08,12.74 GB,2024-01 .. 2025-12,
6,solar_analytics_iceberg,all_uncurtailedpv_v2_flex_included,iceberg,17,,24.0,5.044169e+08,19.46 GB,,
7,solar_analytics_iceberg,all_uncurtailedpv_v2,iceberg,17,,24.0,4.872227e+08,18.81 GB,,
8,sapn2022,circuit_measurements_curtailment_train,iceberg,10,,1.0,4.368811e+08,4.12 GB,,
9,sapn2022,circuit_measurements,iceberg,10,,1.0,3.644280e+08,3.42 GB,,


## 5. `ts` in detail + `is_pv`

- `ts` is partitioned on `(year, month, is_pv)`.
- Every query in the Stage 1 pipeline filters `is_pv = True`. 
- If `ts` has no `is_pv = False` partitions, or they are near-empty, then the load circuits a synthetic meter needs do not exist in this table and Phase 2 has a very different decision to make. If they are substantial, the raw circuit data is there.

In [9]:
ts_tidy = pd.DataFrame()
if ts_key is not None:
    ts_tidy = Inventory.normalise_partitions(raw_partitions[ts_key])
    display(ts_tidy)

    if "is_pv" in ts_tidy.columns and "n_rows" in ts_tidy.columns:
        by_is_pv = (ts_tidy.groupby("is_pv", as_index=False)
                           .agg(n_partitions=("is_pv", "count"),
                                n_rows=("n_rows", "sum"),
                                size_bytes=("size_bytes", "sum")))
        by_is_pv["size"] = by_is_pv.size_bytes.map(Athena.fmt_bytes)
        by_is_pv["share_of_rows"] = (
            by_is_pv.n_rows / by_is_pv.n_rows.sum()
        ).map(lambda v: f"{v:.1%}")
        print("ts rows by is_pv partition:")
        display(by_is_pv[["is_pv", "n_partitions", "n_rows", "size", "share_of_rows"]])
    else:
        print("`$partitions` did not carry an is_pv breakdown with row counts.")
        print("Columns available:", list(ts_tidy.columns))
        print("Phase 2 will have to establish is_pv coverage another way.")


,year,month,is_pv,postcode_bucket,n_rows,n_files,size_bytes
0,2024,1,false,14,26051316,6,869838224
1,2024,1,false,15,17021910,5,554406961
2,2024,1,false,12,20469802,7,694275736
3,2024,1,false,13,24155325,2,786993272
4,2024,1,false,2,26229599,2,851598906
...,...,...,...,...,...,...,...
811,2025,12,true,1,18307702,1,456573565
812,2025,12,true,2,22133544,1,570164595
813,2025,12,true,3,13812026,1,342224164
814,2025,12,true,4,20546137,1,511943718


ts rows by is_pv partition:


,is_pv,n_partitions,n_rows,size,share_of_rows
0,false,408,8521715460,252.02 GB,52.1%
1,true,408,7823538598,204.91 GB,47.9%


### Coverage over time

Rows per month, so gaps and the usable date range are visible rather than assumed.


In [10]:
if len(ts_tidy) and {"year", "month"} <= set(ts_tidy.columns):
    by_month = (ts_tidy.dropna(subset=["year", "month"])
                       .groupby(["year", "month"], as_index=False)
                       .agg(n_rows=("n_rows", "sum") if "n_rows" in ts_tidy.columns
                                     else ("month", "count"),
                            n_partitions=("month", "count")))
    by_month["ym"] = (by_month.year.astype(int).astype(str) + "-"
                      + by_month.month.astype(int).astype(str).str.zfill(2))
    display(by_month[["ym", "n_partitions", "n_rows"]])
    print(f"Coverage: {by_month.ym.iloc[0]} .. {by_month.ym.iloc[-1]} "
          f"({len(by_month)} month-partitions)")
else:
    print("No year/month partition metadata available for ts.")


,ym,n_partitions,n_rows
0,2024-01,34,697961788
1,2024-02,34,675102279
2,2024-03,34,736068248
3,2024-04,34,721252509
4,2024-05,34,747280187
5,2024-06,34,715226154
6,2024-07,34,709630493
7,2024-08,34,706372524
8,2024-09,34,684586254
9,2024-10,34,710778257


Coverage: 2024-01 .. 2025-12 (24 month-partitions)


## 6. Candidate tables

The shortlist is a naming heuristic, it does not exclude anything. 

For each candidate: the schema Athena resolves, then partition coverage. Samples
of the large tables come in section 7, where a partition predicate can be chosen
from what section 5 found.


In [11]:
shortlisted = Inventory.guess_candidates(catalog)
candidates = shortlisted[shortlisted.shortlisted].reset_index(drop=True)
print(f"{len(candidates)} of {len(catalog)} tables shortlisted by name.")
display(candidates[["database", "table", "is_iceberg", "n_columns", "partition_keys"]])


21 of 55 tables shortlisted by name.


,database,table,is_iceberg,n_columns,partition_keys
0,sapn2022,circuit_measurements,True,10,
1,sapn2022,circuit_measurements_curtailment_train,True,10,
2,solar_analytics,circuits,False,7,
3,solar_analytics,meta_single_inverters,False,22,
4,solar_analytics,meta_single_inverters_wrong_capacity,False,37,
5,solar_analytics,meta_single_inverters_wrong_capacity_up2_3c,False,41,
6,solar_analytics,partition_lookup,False,3,
7,solar_analytics,sites,False,16,
8,solar_analytics_iceberg,all_uncurtailedpv,True,8,
9,solar_analytics_iceberg,all_uncurtailedpv_lso,True,8,


In [12]:
for entry in candidates.itertuples(index=False):
    key = f"{entry.database}.{entry.table}"
    schema = columns[(columns.database == entry.database)
                     & (columns.table_name == entry.table)]
    stats = totals.get(key, {})

    # Prefer the ACTUAL partition columns $partitions found over Glue's declared
    # PartitionKeys: Iceberg hides its partition spec from Glue, so `ts` shows
    # "(none)" in `entry.partition_keys` despite genuinely being partitioned --
    # see `Inventory.should_probe_partitions`.
    declared = entry.partition_keys
    actual = ", ".join(stats.get("partition_columns", []) or [])
    if declared:
        partition_desc = declared
    elif actual:
        partition_desc = f"{actual}  [not declared in Glue -- recovered from $partitions]"
    else:
        partition_desc = "(none)"

    print("=" * 78)
    print(f"{key}   [{'iceberg' if entry.is_iceberg else 'hive'}]"
          f"   partitioned by: {partition_desc}")
    if stats.get("n_partitions"):
        print(f"  {stats['n_partitions']} partitions"
              + (f", {stats['n_rows']:,.0f} rows" if stats.get("n_rows") else "")
              + (f", {Athena.fmt_bytes(stats['size_bytes'])}" if stats.get("size_bytes") else "")
              + (f", {stats.get('first_partition')} .. {stats.get('last_partition')}"
                 if stats.get("first_partition") else ""))
    if len(schema):
        display(schema[["ordinal_position", "column_name", "data_type"]]
                .reset_index(drop=True))
    else:
        print("  (no columns resolved via information_schema)")


sapn2022.circuit_measurements   [iceberg]   partitioned by: (none)
  1 partitions, 364,427,972 rows, 3.42 GB
  (no columns resolved via information_schema)
sapn2022.circuit_measurements_curtailment_train   [iceberg]   partitioned by: (none)
  1 partitions, 436,881,072 rows, 4.12 GB
  (no columns resolved via information_schema)
solar_analytics.circuits   [hive]   partitioned by: (none)


,ordinal_position,column_name,data_type
0,1,site_id,bigint
1,2,device_id,bigint
2,3,circuit_id,bigint
3,4,device_type,varchar
4,5,circuit_polarity,integer
5,6,circuit_type,varchar
6,7,is_pv,boolean


solar_analytics.meta_single_inverters   [hive]   partitioned by: (none)


,ordinal_position,column_name,data_type
0,1,circuit_id,bigint
1,2,site_id,bigint
2,3,device_id,bigint
3,4,device_type,varchar
4,5,circuit_polarity,integer
5,6,circuit_type,varchar
6,7,is_pv,boolean
7,8,state,varchar
8,9,postcode,double
9,10,longitude,double


solar_analytics.meta_single_inverters_wrong_capacity   [hive]   partitioned by: (none)


...

,ordinal_position,column_name,data_type
0,1,site_id,bigint
1,2,t_stamp,timestamp(6)
2,3,actual_day,date
3,4,actual_tod,varchar
4,5,v,double
5,6,q_kvar_norm,double
6,7,p_kw_norm,double
7,8,s_norm,double
8,9,ghi,double
9,10,cloud_type,integer


solar_analytics_iceberg.ts   [iceberg]   partitioned by: year, month, is_pv  [not declared in Glue -- recovered from $partitions]
  816 partitions, 16,345,254,058 rows, 456.93 GB, 2024-01 .. 2025-12


,ordinal_position,column_name,data_type
0,1,circuit_id,bigint
1,2,t_stamp,timestamp(6)
2,3,power,double
3,4,energy,double
4,5,energy_reactive,double
5,6,energy_import,double
6,7,energy_export,double
7,8,energy_reactive_import,double
8,9,energy_reactive_export,double
9,10,power_factor,double


solar_analytics_iceberg.voltwatt_uncurtailedpv   [iceberg]   partitioned by: (none)
  1 partitions, 2,873,662 rows, 29.62 MB


,ordinal_position,column_name,data_type
0,1,site_id,bigint
1,2,t_stamp,timestamp(6)
2,3,uncurtailed_p,double
3,4,v,double


## 7. Sample rows

The large tables get a partition predicate taken from the coverage found above,
so this reads one month rather than the table. The dimension tables are small
enough to read directly.

`meta_up23c` is the circuit dimension — one row per circuit. Its column list is
what Phase 3 will build the circuit-to-signal mapping from, so it is worth
reading carefully here even though the taxonomy itself is Phase 3's job.


In [13]:
latest_year = latest_month = None
if len(ts_tidy) and {"year", "month"} <= set(ts_tidy.columns):
    ordered = ts_tidy.dropna(subset=["year", "month"]).sort_values(["year", "month"])
    latest_year = int(ordered.year.iloc[-1])
    latest_month = int(ordered.month.iloc[-1])
    print(f"Sampling ts from the newest partition: year={latest_year} month={latest_month}")
else:
    print("No partition metadata for ts -- skipping the ts sample rather than guessing.")


Sampling ts from the newest partition: year=2025 month=12


In [14]:
if latest_year is not None:
    ts_sample = Athena.aq(
        f"""
        SELECT *
        FROM ts
        WHERE year = {latest_year} AND month = {latest_month}
        LIMIT 5
        """,
        database=Config.SAI,
        label=f"ts sample {latest_year}-{latest_month:02d}",
    )
    display(ts_sample)
    print("Columns:", list(ts_sample.columns))


,circuit_id,t_stamp,power,energy,energy_reactive,energy_import,energy_export,energy_reactive_import,energy_reactive_export,power_factor,voltage,current,year,month,is_pv,postcode
0,692960,2025-12-17 08:30:00,-509.06,-41.0,0.0,0.0,41.0,NaN,NaN,1.0,230.0,-2.21,2025,12,True,2795
1,692960,2025-12-17 09:15:00,-4176.51,-334.0,0.0,0.0,334.0,NaN,NaN,1.0,230.0,-18.16,2025,12,True,2795
2,692960,2025-12-17 09:25:00,-1014.72,-82.0,0.0,0.0,82.0,NaN,NaN,1.0,230.0,-4.41,2025,12,True,2795
3,692960,2025-12-17 08:35:00,-620.29,-51.0,0.0,0.0,51.0,NaN,NaN,1.0,230.0,-2.70,2025,12,True,2795
4,692960,2025-12-17 08:50:00,-553.32,-43.0,0.0,0.0,43.0,NaN,NaN,1.0,230.0,-2.41,2025,12,True,2795


Columns: ['circuit_id', 't_stamp', 'power', 'energy', 'energy_reactive', 'energy_import', 'energy_export', 'energy_reactive_import', 'energy_reactive_export', 'power_factor', 'voltage', 'current', 'year', 'month', 'is_pv', 'postcode']


In [15]:
meta_sample = Athena.aq("SELECT * FROM meta_up23c LIMIT 5", database=Config.SAI,
                   label="meta_up23c LIMIT 5")
display(meta_sample)
print("meta_up23c columns:", list(meta_sample.columns))


,site_id,state,postcode,longitude,latitude,dnsp_name,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,inverter_count,pv_install_date,manufacturer,model,ac_capacity_kw_json,device_id,circuit_id,device_type,circuit_polarity,circuit_type,is_pv,min_time,max_time,v_95,v_05,v_99,v_01,voltage_class,m_id,avg_pf,std_pf,pf_99,pf_01,n_long,n_lat,distance_km,s_99,flex_export_detected
0,1233585204,VIC,3140,145.35,-37.755,Ausnet,5.10,5.0,NaN,2019-08-09,1.0,2019-08-08,Growatt,5000MTL,5.0,124179,219150,Watt Watcher,1,pv_site_net,True,2024-01-01,2024-06-04 20:55:00,241.45218,232.93085,243.20941,230.46495,neutral-absorb,M17,0.998620,0.000225,0.999304,0.998327,145.34,-37.76,1.241018,4.614773,False
1,158521185,NSW,2488,153.55,-28.350,Essential,10.54,8.0,NaN,2019-10-31,1.0,2019-10-30,Sungrow,SG8K-D,8.0,121143,239893,Watt Watcher,1,pv_site_net,True,2024-01-01,2025-06-06 07:10:00,251.09720,242.04462,254.72723,240.55140,absorb,M39,0.999972,0.000029,0.999994,0.999873,153.56,-28.36,1.569777,7.975964,False
2,2083958230,NSW,2763,150.90,-33.750,Endeavour,5.28,5.0,NaN,2020-01-07,1.0,2019-12-13,Generic Inverter,5kW,5.0,105994,254727,Watt Watcher,1,load_air_conditioner,False,2024-01-01,2025-09-11 04:40:00,246.83057,236.57940,248.38918,232.18022,neutral-absorb,M13,0.999996,0.000008,1.000000,0.999959,150.90,-33.74,1.110000,4.930158,True
3,248529252,VIC,3140,145.35,-37.755,Ausnet,5.06,5.0,NaN,2020-03-04,1.0,2020-03-04,Generic Inverter,5kW,5.0,127872,276769,Watt Watcher,1,ac_load_net,False,2024-01-01,2025-09-23 01:10:00,242.80365,235.09694,244.69281,233.32724,neutral-absorb,M13,0.991457,0.006766,0.997926,0.970164,145.34,-37.76,1.241018,3.980736,False
4,248529252,VIC,3140,145.35,-37.755,Ausnet,5.06,5.0,NaN,2020-03-04,1.0,2020-03-04,Generic Inverter,5kW,5.0,127872,276768,Watt Watcher,1,ac_load_net,False,2024-01-01,2025-09-23 01:10:00,242.80365,235.09694,244.69281,233.32724,neutral-absorb,M13,0.991457,0.006766,0.997926,0.970164,145.34,-37.76,1.241018,3.980736,False


meta_up23c columns: ['site_id', 'state', 'postcode', 'longitude', 'latitude', 'dnsp_name', 'dc_capacity_kw', 'ac_capacity_kw', 'export_limit_kw', 'monitoring_start', 'inverter_count', 'pv_install_date', 'manufacturer', 'model', 'ac_capacity_kw_json', 'device_id', 'circuit_id', 'device_type', 'circuit_polarity', 'circuit_type', 'is_pv', 'min_time', 'max_time', 'v_95', 'v_05', 'v_99', 'v_01', 'voltage_class', 'm_id', 'avg_pf', 'std_pf', 'pf_99', 'pf_01', 'n_long', 'n_lat', 'distance_km', 's_99', 'flex_export_detected']


In [16]:
fleet = Inventory.dimension_counts("meta_up23c", database=Config.SAI)
display(fleet)

for table in ("circuits", "sites"):
    try:
        display(Athena.aq(f"SELECT * FROM {table} LIMIT 5", database=Config.SA,
                     label=f"{table} LIMIT 5"))
    except Exception as exc:
        print(f"{table}: {type(exc).__name__}: {str(exc)[:200]}")


,n_rows,n_circuits,n_sites,circuits_per_site
0,60570,60570,16148,3.75


,site_id,device_id,circuit_id,device_type,circuit_polarity,circuit_type,is_pv
0,484720983,135961,84703,Watt Watcher,1,pv_site_net,True
1,484720983,135961,84704,Watt Watcher,1,pv_site_net,True
2,484720983,135961,84705,Watt Watcher,1,pv_site_net,True
3,150652475,140483,658777,Watt Watcher,1,ac_load_net,False
4,150652475,140483,658778,Watt Watcher,1,ac_load_net,False


,site_id,state,postcode,longitude,latitude,dnsp_name,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,inverter_count,pv_install_date,manufacturer,model,ac_capacity_kw_exploaded,installed_after_18_dec_2021
0,1944472430,NSW,2502.0,150.85,-34.470,Endeavour,5.18,5.0,3.0,2021-08-09,1.0,2021-08-09,Sungrow,SG5KTL,5.0,False
1,1245528685,QLD,4078.0,152.95,-27.630,Energex,15.00,10.0,6.8,2024-09-18,1.0,2024-09-18,Generic Inverter,10.0kW,10.0,True
2,1651877625,NSW,2350.0,151.70,-30.510,Essential,11.55,9.6,5.0,2019-06-27,2.0,2018-11-02,Generic Inverter,5kW,5.0,False
3,1555410224,QLD,4814.0,146.75,-19.305,Ergon,13.28,10.0,5.0,2024-03-07,1.0,2024-03-06,Sungrow,SG10RS-ADA,10.0,True
4,623277618,QLD,4211.0,153.30,-27.990,Energex,15.75,10.0,5.0,2024-03-18,1.0,2023-06-30,Fronius,Primo GEN24 10.0,10.0,True


In [17]:
# `partition_lookup` is the pipeline's own record of which partitions exist.
# Worth cross-checking against the $partitions metadata above -- a disagreement
# means one of them is stale.
try:
    lookup = Athena.aq("SELECT * FROM partition_lookup", database=Config.SA,
                  label="partition_lookup")
    print(f"{len(lookup)} rows")
    display(lookup.head(30))
except Exception as exc:
    print(f"partition_lookup: {type(exc).__name__}: {str(exc)[:200]}")


36 rows


,year,month,is_pv
0,2024,1,False
1,2024,1,True
2,2024,10,False
3,2024,10,True
4,2024,11,False
5,2024,11,True
6,2024,12,False
7,2024,12,True
8,2024,2,False
9,2024,2,True


## 8. `structured_data` — the Phase 2 candidate

`ciccada_config.TABLES` maps the logical name to the rebuilt table. Phase 2 decides
whether to build from this or from raw `ts`; this section just establishes what it
is and how big.

Note what `build_structured_data.py` does: it aggregates circuits to **site** level
and filters `is_pv = True` throughout. If that filter has already discarded the
load circuits, this table cannot be the source. **Phase 2 checks that properly** —
here we only record the table's shape.


In [18]:
structured = Config.TABLES["structured_data"]
print(f"structured_data -> {structured}"
      f"   (rebuilt: {'structured_data' in Config.REBUILT})")

structured_key = f"{Config.SAI}.{structured}"
if structured_key in totals:
    print(totals[structured_key])

structured_schema = columns[columns.table_name == structured]
display(structured_schema[["database", "ordinal_position", "column_name", "data_type"]]
        .reset_index(drop=True))


structured_data -> structured_data_v2_flex_included   (rebuilt: True)
{'n_partitions': 24, 'partition_columns': ['year', 'month'], 'n_rows': 871655350.0, 'n_files': 5760.0, 'size_bytes': 39323214947.0, 'year_min': 2024, 'year_max': 2025, 'first_partition': '2024-01', 'last_partition': '2025-12'}


,database,ordinal_position,column_name,data_type
0,solar_analytics_iceberg,1,site_id,bigint
1,solar_analytics_iceberg,2,t_stamp,timestamp(6)
2,solar_analytics_iceberg,3,actual_day,date
3,solar_analytics_iceberg,4,actual_tod,varchar
4,solar_analytics_iceberg,5,v,double
5,solar_analytics_iceberg,6,q_kvar_norm,double
6,solar_analytics_iceberg,7,p_kw_norm,double
7,solar_analytics_iceberg,8,s_norm,double
8,solar_analytics_iceberg,9,ghi,double
9,solar_analytics_iceberg,10,cloud_type,integer


## 9. What this cost

Measured, not estimated. `source = unavailable` means the figure could not be
recovered from the Athena response — the query still ran and was still billed, so
the total is a lower bound in that case.


In [19]:
display(Athena.scan_report())


54 queries, 21.23 MB scanned, ~AUD 0.0041 (billed at a 10.00 MB minimum per query)


,label,database,n_rows,seconds,scanned,scanned_bytes,cost,source
0,information_schema.columns [solar_analytics],solar_analytics,227,3.37,18.94 KB,19390.0,0.0001,query_metadata
1,information_schema.columns [solar_analytics_ic...,solar_analytics_iceberg,445,9.72,40.82 KB,41804.0,0.0001,query_metadata
2,solar$partitions,bom_nci,384,1.96,172.12 KB,176256.0,0.0001,query_metadata
3,elb_logs_tbl$partitions,elb_logdb,29760,2.00,697.50 KB,714240.0,0.0001,query_metadata
4,circuit_measurements$partitions,sapn2022,1,1.87,394 B,394.0,0.0001,query_metadata
5,circuit_measurements_curtailment_train$partitions,sapn2022,1,1.85,394 B,394.0,0.0001,query_metadata
6,test_sola_2025_12$partitions,solar_analytics,184,1.85,4.31 KB,4416.0,0.0001,query_metadata
7,test_sola_2025_7$partitions,solar_analytics,184,1.84,4.31 KB,4416.0,0.0001,query_metadata
8,test_sola_2025_8$partitions,solar_analytics,184,1.84,4.31 KB,4416.0,0.0001,query_metadata
9,test_sola_2025_9$partitions,solar_analytics,184,1.87,4.31 KB,4416.0,0.0001,query_metadata


## 10. Inventory and read on candidates

**Status: CLOSED.** Written from the real output above, not from what the
pipeline code implies. Two low-priority items remain open but do not block
Phase 2 -- see the end of this section.

### Inventory

- **6 Glue databases, 55 tables.** Only 2 are in scope for this project:
  `solar_analytics_iceberg` (35 tables, all Iceberg -- the primary catalogue) and
  `solar_analytics` (13 tables, legacy Hive). `bom_nci` (satellite irradiance,
  8.88B rows) is available but not needed yet.
- **Out of scope, confirmed by name and content, not just naming heuristics:**
  `sapn2022` (a different DNSP's own dataset -- South Australia Power Networks,
  not Solar Analytics), `test_db` (`evm_batch_*` scratch tables), `elb_logdb`
  (AWS load-balancer access logs), and four `solar_analytics.test_sola_2025_*`
  Hive tables (day-partitioned scratch exports).
- **`ts` (`solar_analytics_iceberg.ts`):** 16,345,254,058 rows, ~456.9 GB
  compressed. Partitioned on `(year, month, is_pv)` -- **hidden from Glue's
  declared PartitionKeys**, a real Iceberg-on-Glue quirk, not a data problem
  (see `Inventory.should_probe_partitions`).
- **The `is_pv` split -- the single most consequential number in this phase:**

  | | rows | share | size |
  |---|---:|---:|---:|
  | `is_pv = false` (load) | 8,521,715,460 | 52.1% | 252.0 GB |
  | `is_pv = true` (PV) | 7,823,538,598 | 47.9% | 204.9 GB |

  Load-circuit readings are not a discarded minority -- they are, in raw `ts`,
  slightly the LARGER half of the table. This directly answers the concern in
  the original brief.
- **Fleet size:** 41,393 sites, 171,411 circuits (`circuits` dimension table;
  ~4.14 circuits/site on average). `meta_up23c` has 423,990 rows -- **2.47x
  more than `circuits`, so it is NOT one row per `circuit_id`.**
  `build_structured_data.py` already guards this with `GROUP BY circuit_id` +
  `max(...)` before joining; that convention carries forward rather than being
  rediscovered in Phase 3.
- **`ts` reports 816 total partitions -- RESOLVED.** Not day-level partition
  evolution (my initial guess, and wrong). The raw `$partitions` struct shows a
  FOURTH partition key beyond `(year, month, is_pv)` -- a postcode-bucket
  dimension (exact name truncated in display; confirm with
  `list(ts_tidy.columns)`). Arithmetic proof: 24 months x 2 `is_pv` values x 17
  buckets = 816, exact, and the by-month row counts sum to exactly the total
  row count. `ts` is partitioned at MONTH granularity throughout -- no day-level
  data, no partition-spec evolution. Matters for Phase 4: chunking by
  `(year, month)` alone undercounts the true partition count 17x.
- **Date coverage, confirmed: 2024-01 .. 2025-12, 24 consecutive months, no
  gaps.** (`ami_config.TS_COVERAGE`.)
- **`structured_data` is confirmed PV-only**, independent of any query result:
  its schema has no `is_pv` column at all (none needed -- `ts.is_pv = True` is
  baked into `build_structured_data.py` before the table is written), and its
  partitioning is `(year, month)` only, consistent with a table that only ever
  holds one `is_pv` value. This applies to all three variants seen in the
  catalogue (`structured_data`, `structured_data_v2`,
  `structured_data_v2_flex_included`).
- Every other candidate returned by the naming heuristic
  (`all_uncurtailedpv*`, `conformance_*`, `pv_ghi_norm_model*`, `split_days*`,
  `lso_*`, `voltwatt_uncurtailedpv`) is a DERIVED analytical output of the
  Stage 1/2 pipeline -- modelled PV, conformance verdicts, GHI model artefacts
  -- not raw signal data. Their legacy/`_v2`/`_v2_flex_included` naming pattern
  matches `ciccada_config.REBUILT` exactly (e.g. `conformance_sust_op_3w` and
  `conformance_antiisland` appear ONLY as bare legacy tables, no `_v2` variant
  -- consistent with `ciccada_config`'s own note that those two were not
  rebuilt), which is a good independent cross-check that this inventory and
  the pipeline's own documentation agree.

### Live candidates for a synthetic AMI dataset

1. **`ts` (raw circuit-level telemetry), joined to `meta_up23c`** (site_id,
   circuit_polarity, and whatever `meta_up23c` turns out to carry for circuit
   type -- Phase 3). The only table in the catalogue with circuit-level
   granularity and both PV and load readings. Everything this project needs
   composes from here.

### Ruled out, and why

- **`structured_data` / `_v2` / `_v2_flex_included`** -- site-level, PV-only by
  construction, normalized (`p_kw_norm`, not kW), no circuit breakdown. Cannot
  supply `gross_load` at any granularity.
- **Bare (un-suffixed) `structured_data` and `all_uncurtailedpv`** -- superseded
  legacy tables per `ciccada_config.REBUILT`; not to be used regardless of the
  above.
- **`all_uncurtailedpv*`, `conformance_*`, `pv_ghi_norm_model*`, `split_days*`,
  `lso_*` families** -- derived pipeline outputs, not source signal. (Note for
  later: `all_uncurtailedpv*` -- modelled uncurtailed PV -- could be a useful
  cross-check in Phase 6 for whether the synthetic PV signal looks physically
  plausible. Not needed now.)

### Still open (not blocking Phase 2)

1. Whether `solar_analytics` (legacy Hive)'s `circuits`/`sites` duplicate the
   `solar_analytics_iceberg` ones used above, or differ. Low priority --
   `ciccada_config` documents `solar_analytics_iceberg` as primary -- but worth
   a one-line confirmation before Phase 4 picks a source for the dimension
   tables.
2. The exact name of `ts`'s fourth partition key (truncated in every display
   above as `postcode_bu...`). Cosmetic -- `normalise_partitions` already
   handles it correctly without needing the name -- but worth pinning down
   before Phase 4 writes chunking logic that reasons about it explicitly.

### Carried into Phase 2

- Formalise the recommendation this phase's evidence already points to: raw
  `ts` + `meta_up23c`, not `structured_data`.
- The granularity / cleanliness / cost trade-off table the brief asks for,
  now with real row counts and byte sizes to put in it rather than estimates.
- `ts`'s ~457 GB compressed size is a first data point for Phase 4's extraction
  sizing -- not decided now, just worth carrying forward.
